In [2]:
from tqdm import tqdm, trange
import numpy as np
import pandas as pd
from netCDF4 import Dataset
from pathlib import Path

swot_dir = Path('/nas/cee-ice/data/Confluence_Runs/global_vD/global_vD_mnt/input/swot')
reach_vars = [
    'time',
    'd_x_area',
    'slope',
    'slope_u',
    'slope_r_u',
    'slope2',
    'slope2_u',
    'slope2_r_u',
    'width',
    'p_width',
    'width_u',
    'wse',
    'wse_u',
    'wse_r_u',
    'p_length',
    'reach_q',
    'reach_q_b',
    'dark_frac',
    'ice_clim_f',
    'partial_f',
    'n_good_nod',
    'obs_frac_n',
    'xovr_cal_q',
    'xtrk_dist'
]

In [8]:
svs_ds = Dataset("/nas/cee-water/cjgleason/ted/swot-ml/data/swot_svs/SVS_v1_0.5_trans_v17.nc")
svs_ds

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    title: SWOT Validation Set (SVS)
    summary: Daily in situ river discharge time series and metadata (2023–2025) for SWOT discharge validation.
    creator_name: Peyman Saemian
    contact: Peyman Saemian (saemian@gis.uni-stuttgart.de), Mohammad J. Tourian (tourian@gis.uni-stuttgart.de)
    institution: Institute of Geodesy, University of Stuttgart
    version: 1.0
    license: CC-BY 4.0
    history: Created on 2025-12-12 using MATLAB 25.1.0.2973910 (R2025a) Update 1
    dimensions(sizes): station(3643), time(1096), ymd(3), strlen_station(15), string1(1), strlen_source(7), strlen_rivername(35), num_rchs(8)
    variables(dimensions): float64 Q(station, time), float64 lat(station), float64 lon(station), float64 reach_id_v16(station), int16 date_ymd(ymd, time), |S1 station_id(strlen_station, station, string1), |S1 source(strlen_source, station, string1), |S1 river_name_sword(strlen_rivername, station, string1

In [9]:
svs_reaches = svs_ds['reach_id_v17'][:,0].data
svs_dt = (
    pd.to_datetime(svs_ds['time'][:], unit="D", origin=pd.Timestamp("2023-01-01"))
    + pd.Timedelta(hours=12)        # set to noon
)

mapped_reaches = set()
station_df_list = []
for idx in trange(svs_ds['station'].shape[0]):
    station_reaches = svs_ds['reach_id_v17'][idx,:].data
    valid_reaches = [int(r) for r in station_reaches if np.isfinite(r)]

    reach_data = {}
    for reach_id in valid_reaches:
        reach_file = swot_dir / f"{reach_id}_SWOT.nc"
        if not reach_file.is_file():
            continue
            
        swot_ds = Dataset(reach_file)

        break
    break
#         arrays = [swot_ds['reach'][v][:] for v in reach_vars]  # masked arrays
#         stacked = np.ma.column_stack(arrays)
#         valid_rows = ~np.any(stacked.mask, axis=1)
#         clean_arr = stacked[valid_rows].data

#         # Only keep non-empty
#         if clean_arr.shape[0] > 0:
#             reach_data[reach_id] = clean_arr

#     if not reach_data:
#         # No valid data after applying masks
#         continue
    
        
#     best_reach = max(reach_data, key=lambda rid: reach_data[rid].shape[0])

#     if best_reach in mapped_reaches:
#         print(f"Reach {best_reach} is already in the dataset.")
#         continue
#     mapped_reaches.add(best_reach)
    
#     station_df = pd.DataFrame(reach_data[best_reach], columns = reach_vars)
#     station_df['reach_id'] = best_reach
#     station_df['datetime'] = pd.to_datetime(
#         station_df['time'],
#         unit="s",
#         origin=pd.Timestamp("2000-01-01 00:00:00")
#     )
#     station_df.drop(columns='time', inplace=True)

#     svs_df = pd.Series(data=svs_ds['Q'][idx,:], index=svs_dt).dropna().rename('gage_q')

#     full_df = pd.merge_asof(
#         station_df,
#         svs_df,
#         left_on='datetime',
#         right_index=True,
#         direction='nearest',
#         tolerance=pd.Timedelta('12h')
#     ).dropna()
#     full_df['d_wse'] = full_df['wse'] - full_df['wse'].min()
#     full_df['svs_idx'] = int(svs_ds['station'][idx].data)

#     station_df_list.append(full_df)



# all_stations_df = pd.concat(station_df_list)
# all_stations_df

  0%|          | 0/3643 [00:00<?, ?it/s]


In [15]:
swot_ds['reach']#['hwfit']['fit_coeffs'][:]

<class 'netCDF4.Group'>
group /reach:
    dimensions(sizes): chartime(20)
    variables(dimensions): int64 reach_id(), float64 time(nt), |S1 time_str(nt, chartime), float64 d_x_area(nt), float64 d_x_area_u(nt), float64 slope(nt), float64 slope_u(nt), float64 slope_r_u(nt), float64 slope2(nt), float64 slope2_u(nt), float64 slope2_r_u(nt), float64 width(nt), float64 p_width(nt), float64 width_u(nt), float64 wse(nt), float64 wse_u(nt), float64 wse_r_u(nt), float64 p_length(nt), int32 reach_q(nt), int32 reach_q_b(nt), float64 dark_frac(nt), int32 ice_clim_f(nt), int32 ice_dyn_f(nt), int32 partial_f(nt), int32 n_good_nod(nt), float64 obs_frac_n(nt), int32 xovr_cal_q(nt), float64 xtrk_dist(nt)
    groups: hwfit

In [ ]:
all_stations_df.to_parquet('/home/tlanghorst_umass_edu/cee-water/swot-ml/data/swot_svs/reach_obs.parquet')

In [3]:
all_stations_df = pd.read_parquet('/home/tlanghorst_umass_edu/cee-water/swot-ml/data/swot_svs/reach_obs.parquet')

In [7]:
all_stations_df

,d_x_area,slope,slope_u,slope_r_u,slope2,slope2_u,slope2_r_u,width,p_width,width_u,...,partial_f,n_good_nod,obs_frac_n,xovr_cal_q,xtrk_dist,reach_id,datetime,gage_q,d_wse,svs_idx
0,169.081175,0.000371,0.000023,0.000023,0.000378,0.000003,1.437850e-06,297.891953,298.0,0.748921,...,0.0,61.0,0.772152,0.0,-40626.60156,61670800191,2023-07-26 20:35:44.437999964,506.518,1.3005,2
1,94.023113,0.000382,0.000020,0.000020,0.000437,0.000003,1.446130e-06,457.258145,298.0,2.625897,...,0.0,62.0,0.784810,0.0,8732.97754,61670800191,2023-08-01 07:49:04.365000010,430.725,1.1040,2
2,-85.123099,0.000384,0.000012,0.000012,0.000403,0.000003,7.379900e-07,447.919192,298.0,2.583860,...,0.0,64.0,0.810127,0.0,8593.59668,61670800191,2023-08-22 04:34:09.457999945,215.088,0.6350,2
3,-157.507336,0.000373,0.000023,0.000023,0.000380,0.000003,1.278700e-06,308.628140,298.0,0.863483,...,0.0,64.0,0.810127,0.0,-41286.27344,61670800191,2023-09-06 14:05:56.301000118,141.763,0.4455,2
4,-172.633541,0.000387,0.000021,0.000021,0.000416,0.000003,8.747900e-07,431.725484,298.0,2.307318,...,0.0,61.0,0.772152,0.0,8962.12012,61670800191,2023-09-12 01:19:15.705999851,144.910,0.4059,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,-183.937500,0.000386,0.000023,0.000023,0.000374,0.000004,2.789250e-06,113.395922,101.0,0.930009,...,0.0,40.0,0.769231,0.0,-26441.80469,44560200021,2023-09-04 16:45:43.276999950,137.000,1.8011,3643
2,-684.125000,0.000114,0.000127,0.000127,0.000048,0.000016,1.609442e-05,258.035158,101.0,18.733686,...,1.0,10.0,0.192308,0.0,3138.61560,44560200021,2023-09-18 02:07:18.230999947,171.000,0.0000,3643
3,-137.718750,0.000244,0.000125,0.000125,0.000079,0.000008,7.846700e-06,343.953336,101.0,53.651860,...,1.0,8.0,0.153846,0.0,2915.83960,44560200021,2023-10-08 22:52:19.891000032,133.000,1.9675,3643
4,384.656250,0.000200,0.000025,0.000025,0.000233,0.000004,2.707500e-06,92.848704,101.0,0.870171,...,0.0,47.0,0.903846,0.0,-26937.68359,44560200021,2023-10-16 10:15:50.618999958,241.000,3.8484,3643


In [4]:
prior_ds

NameError: name 'prior_ds' is not defined

In [ ]:
prior_ds['model']

In [15]:
next(prior_files)

'na'

In [16]:
prior_dir = Path("/nas/cee-ice/data/ML_priors/ML_prior_SWORDv17")
prior_files = prior_dir.glob("*.nc")

reach_ids = all_stations_df['reach_id'].unique()



prior_list = []
for prior_file in prior_files:
    prior_ds = Dataset(prior_file)
    
    for reach_id in tqdm(reach_ids, desc=prior_file.stem[0:2]):
        prior_idx = np.where(prior_ds['reaches']['reach_id'][:] == reach_id)[0]
        if prior_idx.shape[0] == 0:
            continue
        elif prior_idx.shape[0] > 1:
            raise ValueError('reach found in multiple locations?')
    
        reach_prior = {
            'reach_id': reach_id,
            'min_q': prior_ds['model']['min_q'][prior_idx][0],
            'mean_q': prior_ds['model']['mean_q'][prior_idx][0],
            'max_q': prior_ds['model']['max_q'][prior_idx][0]
        }
        for i in range(12):
            reach_prior[f'monthly_q_{i+1:02d}'] = prior_ds['model']['monthly_q'][prior_idx, i][0]
        
        prior_list.append(reach_prior)
    
static_df = pd.DataFrame(prior_list)
static_df

af: 100%|██████████| 1281/1281 [00:00<00:00, 1917.13it/s]


,reach_id,min_q,mean_q,max_q,monthly_q_01,monthly_q_02,monthly_q_03,monthly_q_04,monthly_q_05,monthly_q_06,monthly_q_07,monthly_q_08,monthly_q_09,monthly_q_10,monthly_q_11,monthly_q_12
0,61670800191,78.538815,647.824435,3029.773046,467.318096,830.517061,338.632030,1158.167901,2009.782908,1162.722428,734.347837,316.504450,192.475169,124.055882,99.336760,205.847210
1,64289800131,18.642605,60.265080,211.089035,126.313830,96.060803,81.603583,89.979057,50.393696,37.006747,29.905778,26.077921,23.904067,40.960302,63.869517,69.469072
2,62268200731,44.978574,135.645660,649.095690,191.537971,204.260457,160.087709,344.209805,176.737584,108.982575,72.813661,68.724966,65.757986,61.122474,81.740698,111.006750
3,64288000461,4.764050,17.257678,101.924133,21.968484,19.040756,30.640773,26.899306,14.260313,10.327678,8.130038,7.437111,6.710921,20.257485,24.622062,23.401515
4,64288001001,3.335169,6.760528,21.144505,9.650347,9.446388,8.709117,9.441240,6.060902,5.179808,4.625090,4.409538,4.167029,5.547956,7.333653,7.685045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1276,12554000071,2.065799,24.473198,291.186998,62.093512,36.530818,32.479713,51.135857,16.844434,20.578193,6.869769,4.470930,4.064860,9.050719,25.643880,28.876065
1277,12552000731,0.326897,15.911869,344.875193,55.122917,44.021712,16.486397,10.877579,2.886382,1.030479,0.691199,0.502781,0.450914,0.899684,4.700759,54.313716
1278,12551000531,0.867519,10.175399,115.712433,29.226890,14.359155,10.056888,7.485940,4.605154,2.620382,1.689060,1.154632,1.783466,5.318441,11.415763,32.107905
1279,12520700071,7.285938,60.417784,550.090377,169.967553,123.400530,97.617624,67.268313,36.054549,19.061918,14.375973,10.815374,9.333792,12.393987,46.668182,137.930309


In [17]:
static_df = pd.DataFrame(prior_list)

static_df.to_parquet('/nas/cee-water/cjgleason/ted/swot-ml/data/swot_svs/attributes/static.parquet')

In [18]:
list(static_df)

['reach_id',
 'min_q',
 'mean_q',
 'max_q',
 'monthly_q_01',
 'monthly_q_02',
 'monthly_q_03',
 'monthly_q_04',
 'monthly_q_05',
 'monthly_q_06',
 'monthly_q_07',
 'monthly_q_08',
 'monthly_q_09',
 'monthly_q_10',
 'monthly_q_11',
 'monthly_q_12']

In [ ]:
list(all_stations_df)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

test_df = pd.read_parquet('/home/tlanghorst_umass_edu/cee-water/swot-ml/data/swot_svs/reach_obs.parquet')
test_df

In [ ]:
list(test_df)

In [ ]:
all_reaches = test_df['reach_id'].unique()

# shuffle
rng = np.random.default_rng(seed=42)
shuffled = rng.permutation(all_reaches)

# split proportions
train_frac = 0.7
val_frac = 0.15
test_frac = 0.15

n = len(shuffled)
n_train = int(train_frac * n)
n_val = int(val_frac * n)

train_ids = shuffled[:n_train]
val_ids   = shuffled[n_train:n_train + n_val]
test_ids  = shuffled[n_train + n_val:]

# function to write one per line
def write_txt(arr, filename):
    with open(filename, 'w') as f:
        for item in arr:
            f.write(f"{item}\n")

out_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/swot_svs/reach_lists/random")
write_txt(train_ids, out_dir / "train.txt")
write_txt(val_ids, out_dir /  "validate.txt")
write_txt(test_ids, out_dir / "test.txt")